# **Binary Classification1**

### 1) 실습 개요
이번 실습에서는 이진 분류 모델의 학습을 위하여 데이터 전처리를 진행하고, 상황에 맞는 Dataset 클래스와 DataLoader 클래스를 정의하고, 학습을 진행합니다.

### 2) 실습 진행 목적 및 배경
이진 분류 모델은 특징 변수를 바탕으로 사전에 정의된 두 가지 범주 중 하나로 분류하는 예측모델로서, 실생활에서 빈번히 등장하는 문제상황입니다. 따라서 상황에 맞게 특징변수와 목표변수를 전처리하는 것부터 전반적인 이진 분류 모델을 학습시키는 과정에 익숙해지도록 공부합니다.

### 3) 실습 수행으로 얻어갈 수 있는 역량
- 목표 변수를 이산형 레이블로 매핑할 수 있다.
- 특징 변수를 적절히 표준화할 수 있다.
- 상황에 맞는 Dataset클래스를 작성하고, DataLoader객체를 생성할 수 있다.
- 이진 분류 모델을 PyTorch로 구현하고 학습을 진행할 수 있다.

### 4) 실습 핵심 내용
&nbsp;&nbsp; 2.2 트레이닝 데이터<br>
&nbsp;&nbsp; 2.3 Dataset & DataLoader 클래스<br>
&nbsp;&nbsp; 2.4 이진 분류 모델의 학습<br>

### 5) 데이터셋 개요 및 저작권 정보

- 사용 데이터셋: [Iris Dataset](https://www.kaggle.com/datasets/uciml/iris)
  - 붓꽃의 종류(Species), 꽃받침의 길이(SepalLengthCm)와 너비(SepalWidthCm), 꽃잎의 길이(PetalLengthCm)와 너비(PetalWidthCm)이 담긴 데이터셋입니다.
- 저작권 정보: [CC0 1.0 Universal](https://creativecommons.org/publicdomain/zero/1.0/)

### 6) Required Package
```python
torch >= 2.3.0
pandas >= 2.0.3
scikit-learn >= 1.2.2
```

# **2. 이진 분류 모델**

**2.2 트레이닝 데이터**

In [ ]:
import torch

In [ ]:
# Donwload dataset from kaggle
!kaggle datasets download -d uciml/iris
# unzip zip file
!unzip iris.zip

Dataset URL: https://www.kaggle.com/datasets/uciml/iris
License(s): CC0-1.0
  0% 0.00/3.60k [00:00<?, ?B/s]
100% 3.60k/3.60k [00:00<00:00, 5.69MB/s]
Archive:  iris.zip
  inflating: Iris.csv                
  inflating: database.sqlite         


In [ ]:
# 트레이닝 데이터의 코드 표현 실습

import pandas as pd
df = pd.read_csv("Iris.csv", sep = ",", header = 0)[["PetalLengthCm", "Species"]] # 데이터 불러오기

print(df)

     PetalLengthCm         Species
0              1.4     Iris-setosa
1              1.4     Iris-setosa
2              1.3     Iris-setosa
3              1.5     Iris-setosa
4              1.4     Iris-setosa
..             ...             ...
145            5.2  Iris-virginica
146            5.0  Iris-virginica
147            5.2  Iris-virginica
148            5.4  Iris-virginica
149            5.1  Iris-virginica

[150 rows x 2 columns]


In [ ]:
filtered_data = df[df['Species'].isin(['Iris-setosa', 'Iris-versicolor'])]

filtered_df = filtered_data

print(filtered_df)

    PetalLengthCm          Species
0             1.4      Iris-setosa
1             1.4      Iris-setosa
2             1.3      Iris-setosa
3             1.5      Iris-setosa
4             1.4      Iris-setosa
..            ...              ...
95            4.2  Iris-versicolor
96            4.2  Iris-versicolor
97            4.3  Iris-versicolor
98            3.0  Iris-versicolor
99            4.1  Iris-versicolor

[100 rows x 2 columns]


In [ ]:
# 목표 변수를 이산형 레이블로 매핑하는 코드 표현 실습

filtered_df.loc[:, 'Species'] = filtered_df['Species'].map({'Iris-setosa': 0, 'Iris-versicolor': 1})

In [ ]:
# 특징 변수(features)와 목표 변수(target)을 추출하는 코드 표현 실습

x = filtered_df[['PetalLengthCm']].values
t = filtered_df['Species'].values.astype(int)

In [ ]:
# 데이터 분할의 코드 표현 실습

from sklearn.model_selection import train_test_split

x_train, x_test, t_train, t_test = train_test_split(x, t, test_size=0.2, random_state=42)

In [ ]:
# 데이터 표준화의 코드 표현 실습

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [ ]:
# 데이터를 Tensor로 변환하는 코드 표현

x_train = torch.tensor(x_train, dtype=torch.float32)
x_test = torch.tensor(x_test, dtype=torch.float32)
t_train = torch.tensor(t_train, dtype=torch.float32).unsqueeze(1)
t_test = torch.tensor(t_test, dtype=torch.float32).unsqueeze(1)

**2.3 Dataset & DataLoader 클래스**

In [ ]:
# Dataset 클래스의 코드 표현 실습

from torch.utils.data import Dataset, DataLoader

class IrisDataset(Dataset):   # CustomDataset 클래스
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [ ]:
# DataLoader 생성 코드 표현 실습

train_dataset = IrisDataset(x_train, t_train)
test_dataset = IrisDataset(x_test, t_test)

batch_size = 4  # 배치 크기를 4로 설정
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

**2.4 이진 분류 모델**

In [ ]:
# 이진 분류 모델 코드 표현 실습

import torch.nn as nn

class BinaryClassificationModel(nn.Module):
    def __init__(self):
        super(BinaryClassificationModel, self).__init__()
        self.layer_1 = nn.Linear(1, 1)  # 입력 차원과 출력 차원을 1로 설정
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        z = self.layer_1(x)
        y = self.sigmoid(z)
        return y

In [ ]:
# 모델 초기화

model = BinaryClassificationModel()

## 콘텐츠 라이선스

<hr style="height:5px;border:none;color:#5F71F7;background-color:#5F71F7">

<font color='red'><b>WARNING</font> : 본 교육 콘텐츠의 지식재산권은 재단법인 네이버커넥트에 귀속됩니다. 본 콘텐츠를 어떠한 경로로든 외부로 유출 및 수정하는 행위를 엄격히 금합니다. 다만, 비영리적 교육 및 연구활동에 한정되어 사용할 수 있으나 재단의 허락을 받아야 합니다. 이를 위반하는 경우, 관련 법률에 따라 책임을 질 수 있습니다. </b>